In [ ]:
from dotenv import load_dotenv
#import os

load_dotenv()  # Loads variables from .env file into os.environ

#os.environ["OPENAI_API_KEY"] = "<OPENAI_API_KEY harcoded here>"
openai_api_key = os.getenv("OPENAI_API_KEY")
if not openai_api_key:
    raise ValueError("OPENAI_API_KEY not found in .env file")



In [ ]:
from openai import OpenAI

openai_client = OpenAI()

model = "text-embedding-3-large"

def get_embeddings(text, input_type="document"): #
    response = openai_client.embeddings.create(
        input=text,
        model=model
    )
    print(response.data[0])
    return response.data[0].embedding

I couldn't find the function in your files, but based on the context (advanced RAG project), this is almost certainly using the **Cohere Embeddings API** — and I can fully explain why `input_type="document"` is there.

---

## Why `input_type="document"` exists

Cohere's embedding API requires you to tell it **how the text will be used**, so it can optimize the embedding accordingly. There are two main values:

| `input_type` | Used when... |
|---|---|
| `"document"` | Embedding text that will be **stored/indexed** (e.g., your knowledge base chunks) |
| `"search_query"` | Embedding a **user's question** to search against those documents |

The function signature:

```python
def get_embeddings(text, input_type="document"):
```

sets `"document"` as the **default**, meaning: *"unless told otherwise, assume we're embedding a chunk to be stored."*

---

## Why does Cohere need to know this?

Cohere uses **asymmetric embedding** — the document embedding space and query embedding space are trained separately so that:

- A **query** like *"what is RAG?"* maps close to...
- A **document** chunk like *"RAG stands for Retrieval-Augmented Generation..."*

...even though the two texts look very different. Without this distinction, the dot-product similarity scores would be less accurate.

---

## In practice in a RAG pipeline:

```python
# Indexing phase — embed your documents
doc_embedding = get_embeddings(chunk, input_type="document")  # default

# Query phase — embed the user's question
query_embedding = get_embeddings(user_question, input_type="search_query")  # override
```

So the default `"document"` makes sense because **most calls** to this function happen during indexing. The query embedding call is the special case that overrides it.

In [ ]:
get_embeddings("Hello World")

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = PyPDFLoader("https://investors.mongodb.com/node/12236/pdf")
data = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=20)
documents =text_splitter.split_documents(data)

In [ ]:
docs_to_insert = [{
    "text" : doc.page_content,
    "embeddings" : get_embeddings(doc.page_content)
} for doc in documents ]

In [ ]:
import os
from dotenv import load_dotenv
from pymongo import MongoClient

load_dotenv()  # Load variables from .env

mongo_uri = os.getenv("MONGODB_URI")
if not mongo_uri:
    raise ValueError("MONGODB_URI not found in .env file")

mongo_client = MongoClient(mongo_uri)
collection = mongo_client["rag_db"]["searchable_docs"]

## Breaking down `mongo_client["ragdb"]["searchable_docs"]`

```python
collection = mongo_client["ragdb"]["searchable_docs"]
```

This line uses **two chained subscript operators** on a MongoDB client:

| Position | Value | What it refers to |
|---|---|---|
| 1st `["ragdb"]` | `ragdb` | The **Database** name |
| 2nd `["searchable_docs"]` | `searchable_docs` | The **Collection** name (like a table in SQL) |

So it's equivalent to saying: *"Give me the `searchable_docs` collection inside the `ragdb` database."*

---

## Will they be created when this cell runs?

**No — not yet.** This line only creates a **reference** (a Python object pointing to where they would live). MongoDB uses **lazy creation**:

- The database `ragdb` is **not created** yet
- The collection `searchable_docs` is **not created** yet

> **They are only physically created in MongoDB when you first insert a document into `collection`.**

```python
# Only at this point do "ragdb" and "searchable_docs" actually get created:
collection.insert_one({"key": "value"})
```

This is a core MongoDB concept — it won't complain or throw an error when you reference a non-existent database or collection. It just waits until you actually write data.

In [ ]:
result=collection.insert_many(docs_to_insert)

Here's the same content formatted as clean Markdown for a notebook cell:

````markdown
## 🗄️ MongoDB Vector Search — How It Works

### Collection Structure

The `searchable_docs` collection stores each document with two key fields:

| Field       | Type        | Example                            |
|-------------|-------------|------------------------------------|
| `text`      | string      | `"MongoDB is a NoSQL database..."` |
| `embedding` | float array | `[0.1, 0.4, ..., 3072 dims]`       |

A **vector index** (`vector_index`) is built on the `embedding` field — this enables fast semantic similarity search across all documents.

---

### ⚡ Why the Index Is Required

| | Without `vector_index` | With `vector_index` (HNSW) |
|---|---|---|
| **Strategy** | Brute-force scan every document | Navigate pre-built graph to nearest cluster |
| **Speed** | O(N) — slows with every new doc ❌ | O(log N) — fast even at millions of docs ✅ |
| **At 1M docs** | Unusably slow ❌ | Near-instant ✅ |
| **`$vectorSearch`** | Hard-fails with `OperationFailure` ❌ | Works correctly ✅ |

---

### 🔍 What the Index Enables

> **"Find the documents whose `embedding` is closest to this query embedding"**  
> — measured using **cosine similarity** (angle between vectors)

```
User Question  →  get_embeddings()  →  [0.12, 0.87, ..., 3072 numbers]
                                                    │
                                         $vectorSearch via HNSW
                                                    │
                                        ┌───────────┴──────────┐
                                        │  Pre-select 100 docs │
                                        │  Re-rank by cosine   │
                                        │  Return top-5 docs   │
                                        └──────────────────────┘
```
````

---

Just paste the above into a **Markdown cell** in your notebook. Here's what each section gives you:

| Element        | Purpose |
|----------------|---------|
| `##` heading   | Clear section title |
| First table    | Shows the document structure stored in MongoDB |
| Second table   | Side-by-side index vs. no-index comparison |
| `> blockquote` | Highlights the key concept visually |
| Code block     | Shows the vector search flow cleanly |

In [ ]:
from pymongo.operations import SearchIndexModel
import time

# ═════════════════════════════════════════════════════════════════════
# WHY DO WE NEED AN INDEX FOR SEMANTIC / VECTOR SEARCH?
#
# Each document in MongoDB contains an "embedding" field — a list of
# 3072 floating-point numbers representing the MEANING of that text.
#
# A semantic search query works like this:
#   1. Convert user's question → query embedding (3072 numbers)
#   2. Find documents whose embeddings are "closest" to the query
#      embedding using cosine similarity
#
# The naive approach (no index) = BRUTE-FORCE SCAN:
#   → Load every document from disk
#   → Compute cosine similarity against every embedding
#   → Sort all results
#   → Return top-k
#
# This is O(N) — it gets slower with every new document added.
# At 1 million docs, this becomes unusably slow.
#
# A Vector Index uses an algorithm called HNSW (Hierarchical Navigable
# Small World) — a graph-based structure that groups similar vectors
# together, so at query time MongoDB only checks a SMALL SUBSET of
# documents rather than all of them.
#
# This makes search O(log N) — extremely fast even at millions of docs.
# ═════════════════════════════════════════════════════════════════════

# ─────────────────────────────────────────────────────────────
# CAN WE JUST QUERY THE COLLECTION DIRECTLY WITHOUT AN INDEX?
#
# For regular field lookups (e.g. find by name, date, ID) → YES.
# MongoDB can still scan documents field by field.
#
# For vector/semantic search using $vectorSearch aggregation → NO.
# MongoDB Atlas REQUIRES a vector search index to exist on the
# "embedding" field. Without it:
#
#   OperationFailure: "Search index 'vector_index' does not exist"
#
# The $vectorSearch pipeline stage is NOT a simple filter — it
# requires the pre-built HNSW graph to navigate efficiently.
# There is NO fallback to a brute-force scan. It will hard-fail.
#
# Think of it like this:
#   Regular query = reads a book page by page (slow but possible)
#   Vector search = needs a pre-built map of the book to navigate
#                   (impossible without the map)
# ─────────────────────────────────────────────────────────────

# ─────────────────────────────────────────────────────────────
# STEP 1: Define the name of the index we want to create
# ─────────────────────────────────────────────────────────────
index_name = "vector_index"

# ─────────────────────────────────────────────────────────────
# STEP 2: Define the structure of the Vector Search Index
#
# SearchIndexModel tells MongoDB Atlas HOW to index your data
# for vector (semantic) similarity searches.
# ─────────────────────────────────────────────────────────────
search_index_model = SearchIndexModel(
  definition = {
    "fields": [
      {
        # This field holds the embedding vectors stored in each document
        "type": "vector",

        # Must match the output dimension of your embedding model.
        # CRITICAL: If this number is wrong, the index will be built
        # incorrectly and your similarity scores will be meaningless.
        # e.g. OpenAI "text-embedding-3-large" outputs 3072 dimensions
        "numDimensions": 3072,

        # The key name in your MongoDB document that stores the embedding.
        # e.g. each doc looks like: { "text": "...", "embedding": [...] }
        # The index is built SPECIFICALLY on this field — other fields
        # are ignored by the vector index.
        "path": "embedding",

        # Similarity metric used to compare two vectors at query time.
        # "cosine" = measures the ANGLE between vectors, ignoring magnitude.
        #            Best for text — robust even if embedding lengths vary.
        # "dotProduct" = magnitude matters; vectors must be pre-normalized.
        # "euclidean"  = measures straight-line distance between vectors.
        "similarity": "cosine"
      }
    ]
  },
  name = index_name,    # Label this index as "vector_index"
  type = "vectorSearch" # Tells Atlas this is a vector search index (not a regular text index)
)

# ─────────────────────────────────────────────────────────────
# STEP 3: Submit the index creation request to MongoDB Atlas
#
# This triggers Atlas to START building the index in the background.
# Atlas reads all existing documents in the collection, extracts the
# "embedding" field from each, and builds the HNSW graph structure.
#
# It does NOT block — the index is NOT immediately ready after this line.
# Any $vectorSearch query attempted now will FAIL with an error.
# ─────────────────────────────────────────────────────────────
collection.create_search_index(model=search_index_model)


# ─────────────────────────────────────────────────────────────
# STEP 4: Poll MongoDB Atlas until the index is ready
#
# Atlas builds vector indexes asynchronously (can take ~30-60 seconds).
# We MUST wait until Atlas reports it as "queryable = True" before
# running any $vectorSearch queries — otherwise they will hard-fail.
#
# WHAT HAPPENS IF YOU QUERY TOO EARLY (before index is ready)?
#   → OperationFailure: "$vectorSearch" is not ready / does not exist
#   → Your RAG pipeline would crash at the retrieval step
#   → This polling loop is the safety gate that prevents that.
# ─────────────────────────────────────────────────────────────
print("Polling to check if the index is ready. This may take up to a minute.")

predicate = None

# Default readiness check: the index is ready when "queryable" == True
# "queryable" is a status flag Atlas sets once the HNSW graph is fully
# built and verified — only then can it serve $vectorSearch queries.
if predicate is None:
   predicate = lambda index: index.get("queryable") is True

# Poll every 5 seconds until the index becomes queryable
while True:
   # Ask Atlas for the current build status of our named index
   indices = list(collection.list_search_indexes(index_name))

   # If the index exists AND passes the readiness check → exit the loop
   if len(indices) and predicate(indices[0]):
      break

   time.sleep(5)  # Wait 5 seconds before checking again (avoid hammering Atlas)

print(index_name + " is ready for querying.")
# ─────────────────────────────────────────────────────────────
# After this line, $vectorSearch queries using "vector_index"
# will work correctly and return semantically similar documents.
# ─────────────────────────────────────────────────────────────


In [ ]:
query_embeddings = get_embeddings("What is MongoDB?")

In [ ]:
# ═════════════════════════════════════════════════════════════════════
# THIS CELL: Performs a Semantic / Vector Similarity Search
#
# Goal: Find the 5 documents in the collection whose meaning is
#       most similar to the question "What is MongoDB?"
#
# How it works:
#   1. The question is converted to a 3072-dim embedding vector
#      using get_embeddings()
#   2. MongoDB Atlas compares that query vector against all stored
#      "embedding" fields using the pre-built HNSW vector index
#   3. It returns the top-k most similar documents by cosine distance
# ═════════════════════════════════════════════════════════════════════

# NOTE: This looks like a bug — "collection" already IS "searchable_docs"
# (set earlier as: collection = mongo_client["ragdb"]["searchable_docs"])
# So this line is calling .searchable_docs.aggregate() on the collection
# object, which will likely fail. It should just be:
#   collection.aggregate([...])
collection.searchable_docs.aggregate([

    {
        # $vectorSearch is a MongoDB Atlas aggregation stage — it is NOT
        # available in a self-hosted MongoDB. It requires MongoDB Atlas
        # with a vector index already built and in "queryable" state.
        "$vectorSearch": {

            # The name of the vector index to use for this search.
            # Must match exactly what was created in the previous cell.
            "index": "vector_index",

            # The field in each document that holds the stored embeddings.
            # Atlas will compare the queryVector against this field.
            "path": "embedding",

            # ─────────────────────────────────────────────────────────
            # queryVector: The embedding of the QUESTION being searched.
            #
            # get_embeddings("What is MongoDB?") converts the string
            # into a 3072-dim vector using the same embedding model
            # that was used to embed the documents at indexing time.
            #
            # CRITICAL: You must use the SAME embedding model for both
            # documents and queries — mixing models breaks similarity.
            #
            # The commented-out line below shows the intended pattern:
            #   query_embeddings = get_embeddings(user_question)
            # and then pass the pre-computed variable. Here it's done
            # inline for testing/simplicity.
            # ─────────────────────────────────────────────────────────
            #"queryVector": query_embeddings,   # ← production pattern
            "queryVector": get_embeddings("What is MongoDB?"),  # ← inline for testing

            # ─────────────────────────────────────────────────────────
            # numCandidates: How many documents Atlas pre-selects from
            # the HNSW index BEFORE applying the final re-ranking.
            #
            # A larger value = more accurate results but slower.
            # Rule of thumb: numCandidates should be 10x-20x "limit".
            # Here: 100 candidates → narrowed down to 5 final results.
            # ─────────────────────────────────────────────────────────
            "numCandidates": 100,

            # The final number of top results to return to the caller.
            # These are the 5 documents most semantically similar to
            # "What is MongoDB?" — to be used as context for the LLM.
            "limit": 5
        }
    }
])


In [ ]:
# ═════════════════════════════════════════════════════════════════════
# THIS CELL: Defines the core RAG Retrieval Function
#
# get_query_results() is the RETRIEVAL step of the RAG pipeline.
# Given a user's question (plain text), it:
#   1. Converts the question into a vector embedding
#   2. Runs a semantic similarity search against MongoDB Atlas
#   3. Returns the top 5 most relevant document texts as a list
#
# These returned texts are then used as CONTEXT for the LLM to
# generate a grounded, factual answer.
# ═════════════════════════════════════════════════════════════════════

def get_query_results(query):
  """Gets results from a vector search query."""

  # ─────────────────────────────────────────────────────────────
  # STEP 1: Convert the user's question into a vector embedding.
  #
  # input_type="query" is crucial here — it tells the embedding
  # model that this text is a SEARCH QUERY, not a document.
  # Cohere (and similar models) use different internal weights
  # for queries vs. documents to improve retrieval accuracy.
  #
  # This must use the SAME embedding model used during indexing,
  # otherwise the vector spaces won't align and results will be wrong.
  # ─────────────────────────────────────────────────────────────
  query_embedding = get_embeddings(query, input_type="query")
  print(query_embedding)  # Debug: shows the raw 3072-dim vector

  # ─────────────────────────────────────────────────────────────
  # STEP 2: Build the MongoDB Aggregation Pipeline
  #
  # An aggregation pipeline is a sequence of stages that MongoDB
  # processes one after another — like a data processing chain.
  # Each stage transforms the documents before passing to the next.
  # ─────────────────────────────────────────────────────────────
  pipeline = [

      # ── STAGE 1: $vectorSearch ────────────────────────────────
      # Performs the semantic similarity search using the HNSW index.
      # Finds documents whose "embedding" field is closest (by cosine
      # similarity) to our query_embedding.
      {
            "$vectorSearch": {
              "index": "vector_index",       # The pre-built HNSW vector index
              "queryVector": query_embedding, # Our question as a vector
              "path": "embedding",            # Field in docs holding stored vectors

              # ⚠️ LIKELY BUG: numCandidates=3072 looks like a copy-paste
              # error — 3072 is the embedding dimension, not a candidate count.
              # numCandidates controls how many docs the HNSW graph pre-selects
              # before final re-ranking. Recommended: 10x–20x the "limit".
              # For limit=5, a value of 50–100 is appropriate, not 3072.
              # Using 3072 here means Atlas will scan far more candidates than
              # needed, making the query slower than necessary.
              "numCandidates": 3072,  # ← should likely be 50 or 100

              "limit": 5  # Return only the top 5 most similar documents
            }
      },

      # ── STAGE 2: $project ────────────────────────────────────
      # Controls which fields are returned in the final output.
      # Acts as a "SELECT" clause in SQL — include/exclude fields.
      {
            "$project": {
              "_id": 0,   # 0 = EXCLUDE: suppress MongoDB's internal document ID
              "text": 1   # 1 = INCLUDE: return only the "text" field of each doc
              # Note: the "embedding" vector is NOT returned — it's large (3072
              # floats) and not needed by the LLM. Only the raw text is useful.
         }
      }
  ]

  # ─────────────────────────────────────────────────────────────
  # STEP 3: Execute the pipeline against the MongoDB collection.
  #
  # collection.aggregate() sends the pipeline to Atlas and returns
  # a Cursor — a lazy iterator that fetches results on demand.
  # It does NOT immediately load all results into memory.
  # ─────────────────────────────────────────────────────────────
  results = collection.aggregate(pipeline)
  print(results)  # Debug: prints the Cursor object (not the actual data yet)

  # ─────────────────────────────────────────────────────────────
  # STEP 4: Materialise the Cursor into a plain Python list.
  #
  # We iterate over the cursor to force-fetch all results from Atlas
  # and collect them into a list so they can be reused multiple times.
  #
  # A Cursor can only be iterated ONCE — converting to a list lets
  # us pass the results to the LLM prompt without re-querying Atlas.
  # ─────────────────────────────────────────────────────────────
  array_of_results = []
  for doc in results:
      array_of_results.append(doc)  # Each doc = {"text": "..."}

  # Returns e.g.: [{"text": "MongoDB is..."}, {"text": "Atlas is..."}, ...]
  # These 5 texts will be injected as context into the LLM prompt.
  return array_of_results


In [ ]:
get_query_results("mongodb vector search")